In [2]:
using DelimitedFiles
include("open_optimization_problem.jl")   # pulls in the full include chain

n      = 3
J      = fill(1/4, n - 1)
gammas = fill(J[1]/4, n)
tlist  = range(0, 25; length = 100)
excited = ["0"]
ks     = [3, 8, 12]
dissipation = false

cutoff, maxdim = 0.0, 16     # was 1e-10, 200 — maxdim=16 is already exact for n=3
order     = 2     # order of the product formulas being combined
k_ref     = 100   # fine reference standing in for e^{tL} in F_ex
order_ref = 4                # was 2

lsites = liouville_siteinds(n)
rho0   = vectorized_initial_state_mps(lsites, excited)

coeffs = zeros(Float64, length(tlist), length(ks))

for (i, t) in enumerate(tlist)
    if t <= 0
        coeffs[i, :] .= NaN
        continue
    end
    M, _ = open_gram_matrix(n, J, gammas, t, ks, lsites, rho0;
                            cutoff = cutoff, maxdim = maxdim,
                            order = order, dissipation = dissipation)
    L, _ = open_L_vector(n, J, gammas, t, ks, k_ref, lsites, rho0;
                         cutoff = cutoff, maxdim = maxdim,
                         order = order, order_ref = order_ref,
                         dissipation = dissipation)
    c, _ = dynamic_mpf_coefficients(M, L)
    coeffs[i, :] .= c
    println("t = ", round(t, digits = 4), "  c = ", c,
            "  sum = ", sum(c), "  cond(M) = ", cond(M))
end

open("n_3_mpf_coefficients.txt", "w") do io
    println(io, "# t\tc_k3\tc_k8\tc_k12")
    writedlm(io, hcat(collect(tlist), coeffs))
end

t = 0.2525  c = [-0.26997336777990144, 2.440781416499115, -1.1708080487192134]  sum = 1.0000000000000002  cond(M) = 2.3593047268616762e14
t = 0.5051  c = [-0.39619490081557274, 3.9549482658135098, -2.558753364997937]  sum = 1.0  cond(M) = 1.879908231395354e14
t = 0.7576  c = [-0.46885409814426454, 4.827466464519678, -3.358612366375413]  sum = 1.0000000000000004  cond(M) = 5.753099601658005e14
t = 1.0101  c = [0.01664101377761853, -0.9997158350721925, 1.983074821294574]  sum = 0.9999999999999999  cond(M) = 1.7448139720680712e14
t = 1.2626  c = [0.032560615788947755, -1.1908602062052995, 2.1582995904163518]  sum = 1.0  cond(M) = 6.550625187037396e13
t = 1.5152  c = [0.014393718545688257, -0.9727504313249744, 1.9583567127792862]  sum = 1.0  cond(M) = 8.996821048931354e12
t = 1.7677  c = [0.01160013682232841, -0.939203060010154, 1.9276029231878256]  sum = 1.0  cond(M) = 1.5157868193708828e12
t = 2.0202  c = [0.0112268597174598, -0.9347160757626292, 1.9234892160451693]  sum = 1.0  cond(M) =